# Four ways an intrusion-detection score lies

*Accuracy, shortcut features, shuffled splits and preprocessing before the split, measured on the same data.*

Most intrusion-detection results on lab captures look excellent. Many of them do not survive contact with new traffic, because the evaluation let information from the test set leak into training, or let the model read the answer from a column that will not exist in production.

This notebook takes the [Cyber Attack Dataset: ARP, SYN, PING Flood (v3)](https://www.kaggle.com/datasets/aleksandarraki/cyber-attack-dataset-arp-syn-ping-flood) and measures four of those traps one by one. Each section changes one thing and keeps everything else fixed.

| # | Trap | What it does to the score here |
|---|------|------|
| 1 | Reporting accuracy on imbalanced data | a model that never detects an attack scores 98 % |
| 2 | Shortcut features | the frame number alone catches most attack packets on a shuffled split |
| 3 | Shuffled split of time-ordered traffic | hides that ARP storm detection breaks on later bursts |
| 4 | Fitting preprocessing on all rows | small here, but it is the flaw that version 2 of this dataset had |

Every number quoted comes from the outputs below. Exact values move slightly between scikit-learn versions (a run on Kaggle and a local run differ in the second or third decimal); the conclusions do not.

In [ ]:
import glob
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import sklearn

print("scikit-learn", sklearn.__version__)
SEED = 42
LABELS = ["Normal", "ARP_Spoofing", "ARP_Storm", "PING_Flood", "SYN_Flood"]


def find(name):
    """Locate a dataset file on Kaggle (/kaggle/input/...) or next to the notebook."""
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) or glob.glob(f"**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]


full = pd.read_csv(find("network_attack_traffic.csv"))
print(f"{len(full):,} packets, {full['label'].nunique()} classes")
full["label_name"].value_counts().to_frame("packets")

The dataset already ships a **chronological split**: for each class, the first 80 % of its packets in capture order are training data and the last 20 % are test data. Below it is compared with the common alternative, a **random stratified split** of the same rows.

In [ ]:
FLAGS = ["tcp_flag_fin", "tcp_flag_syn", "tcp_flag_rst", "tcp_flag_psh", "tcp_flag_ack"]
HEADER = ["frame_time_delta", "arp_opcode", "icmp_type_code", "tcp_seq", "tcp_hdr_len", "data_len"] + FLAGS


def encode(df):
    """Numeric matrix: ARP opcode and device roles become integer codes, missing values stay NaN."""
    x = df.copy()
    if "arp_opcode" in x:
        x["arp_opcode"] = x["arp_opcode"].map({"request": 1, "reply": 2})
    for col in x.columns.intersection(["arp_src_device", "arp_dst_device"]):
        x[col] = x[col].map({"attacker": 0, "victim": 1, "unresolved": 2,
                             "other_1": 3, "other_2": 4, "other_3": 5, "other_4": 6})
    return x.astype(float)


chrono = (full[full["split"] == "train"], full[full["split"] == "test"])
shuffled = train_test_split(full, test_size=0.2, stratify=full["label"], random_state=SEED)
SPLITS = {"chronological": chrono, "random": shuffled}


def evaluate(model, cols, split):
    train, test = SPLITS[split]
    model.fit(encode(train[cols]), train["label"])
    pred = model.predict(encode(test[cols]))
    recall = recall_score(test["label"], pred, average=None, labels=range(5))
    return {"split": split,
            "accuracy": accuracy_score(test["label"], pred),
            "macro_f1": f1_score(test["label"], pred, average="macro"),
            **{f"recall_{name}": r for name, r in zip(LABELS, recall)}}


def gbm():
    return HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED)


for name, (train, test) in SPLITS.items():
    print(f"{name:14s} train {len(train):,}  test {len(test):,}")

## 1. Accuracy on imbalanced traffic

98.2 % of the packets are normal. A model that never raises an alert is therefore 98 % accurate. **Macro-F1** averages the F1 score of every class with equal weight, so it cannot be carried by the majority class.

In [ ]:
dummy = evaluate(DummyClassifier(strategy="most_frequent"), HEADER, "chronological")
print(f"'Always normal': accuracy {dummy['accuracy']:.4f}, macro-F1 {dummy['macro_f1']:.4f}, "
      f"attacks detected: {sum(dummy[f'recall_{c}'] for c in LABELS[1:]):.0f} of 4 classes")

Every score below is macro-F1 plus the recall of each attack class. Accuracy on this dataset says almost nothing.

## 2. Shortcut features

`frame_number` is the position of the packet in the capture, and the attacks were run at different times. A model can learn *when* each attack happened instead of *what* it looks like. The device-role columns (`arp_src_device`, `arp_dst_device`) are similar: in this lab, the attacker is always the same machine.

Neither will be true on another network, so they are shortcuts. Here is a model that sees **only the frame number**:

In [ ]:
rows = [evaluate(gbm(), ["frame_number"], split) for split in SPLITS]
table_fn = pd.DataFrame(rows).set_index("split")[["macro_f1"] + [f"recall_{c}" for c in LABELS[1:]]]
table_fn.round(3)

In [ ]:
for c in LABELS[1:]:
    r, ch = table_fn.loc["random", f"recall_{c}"], table_fn.loc["chronological", f"recall_{c}"]
    print(f"{c:13s} recall {r:.2f} with a random split, {ch:.2f} with the chronological split")

On the random split, the frame number alone catches most attack packets, because every test packet sits between training packets of the same attack burst. It also raises many false alarms, which is why its macro-F1 stays low. On the chronological split, where the test packets come from later in the capture, recall falls for the ARP and PING attacks. SYN flood holds up only because it is the last attack in the capture, so "late frame number" happens to mean "SYN flood".

Adding the shortcuts to the honest header features hides the problem completely:

In [ ]:
SHORTCUTS = ["frame_number", "arp_src_device", "arp_dst_device"]
rows = []
for split in SPLITS:
    for label, cols in {"header only": HEADER, "header + shortcuts": HEADER + SHORTCUTS}.items():
        rows.append({"features": label, **evaluate(gbm(), cols, split)})
table2 = pd.DataFrame(rows).set_index(["features", "split"])
table2[["macro_f1"] + [f"recall_{c}" for c in LABELS[1:]]].round(4)

With the shortcuts, every setting scores close to 1.0 and the comparison between splits becomes meaningless. Everything below uses the **header-only** features.

## 3. Shuffled split of time-ordered traffic

Consecutive packets of one attack are nearly identical. With a random split, almost every test packet has a twin a few frames away in the training data, so the test set measures memory rather than generalization.

The chronological split asks the realistic question: *trained on the earlier part of each attack, does the model recognize the later part?*

In [ ]:
rows = [evaluate(gbm(), HEADER, split) for split in SPLITS]
table3 = pd.DataFrame(rows).set_index("split")[["macro_f1"] + [f"recall_{c}" for c in LABELS[1:]]]
table3.round(4)

The overall difference looks small, but it is concentrated in one class: on the random split ARP storm detection looks perfect, while on the chronological split part of the later storm traffic is missed. A random split would have hidden the one weakness worth investigating.

The timeline shows why. ARP storm packets come in several bursts across the capture, and the bursts at the end are not the same as the ones at the start:

In [ ]:
attacks = full[full["label"] > 0]
fig, ax = plt.subplots(figsize=(11, 3.2))
for i, name in enumerate(LABELS[1:], start=1):
    part = attacks[attacks["label"] == i]
    colors = np.where(part["split"] == "train", "#1f6feb", "#f0883e")
    ax.scatter(part["frame_number"], np.full(len(part), i), s=4, c=colors, alpha=0.6)
ax.set_yticks(range(1, 5), LABELS[1:])
ax.set_xlabel("frame number (capture order)")
ax.set_title("Attack packets over the capture: training part (blue) and test part (orange) of each class")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Preprocessing fitted on all rows

Version 2 of this dataset was standardized with the mean and standard deviation of **all** rows, missing values were filled with the mean of all rows, and only then was the data split. The test rows thereby influenced the training features.

To isolate that effect, the next cell trains the same logistic regression twice on the chronological split:
- **leaky:** the imputer and scaler are fitted on train and test together (the version 2 procedure),
- **correct:** the imputer and scaler are fitted on the training rows only (a scikit-learn pipeline does this automatically).

In [ ]:
def logreg():
    return LogisticRegression(class_weight="balanced", max_iter=2000)


train, test = chrono
x_train, x_test = encode(train[HEADER]), encode(test[HEADER])

# Leaky: statistics from all rows, as in version 2
imputer = SimpleImputer(strategy="mean").fit(pd.concat([x_train, x_test]))
scaler = StandardScaler().fit(imputer.transform(pd.concat([x_train, x_test])))
leaky = logreg().fit(scaler.transform(imputer.transform(x_train)), train["label"])
pred_leaky = leaky.predict(scaler.transform(imputer.transform(x_test)))

# Correct: every fitted step sees the training rows only
pipe = make_pipeline(SimpleImputer(strategy="mean"), StandardScaler(), logreg()).fit(x_train, train["label"])
pred_ok = pipe.predict(x_test)

table4 = pd.DataFrame([
    {"preprocessing": "fitted on all rows (v2)", "macro_f1": f1_score(test["label"], pred_leaky, average="macro")},
    {"preprocessing": "fitted on training rows", "macro_f1": f1_score(test["label"], pred_ok, average="macro")},
]).set_index("preprocessing")
table4.round(4)

On this dataset the difference is small: the test rows are only a fifth of the data and the feature distributions are stable, so the leaked statistics barely move. That is the honest result, and it is also why this kind of leakage survives review: it rarely shows up as an obvious jump. It becomes large when the test set is a different network, a different day, or a shifted distribution, which is exactly the situation an IDS is built for. The fix costs nothing: put every fitted step in a pipeline.

## 5. How far the honest number can be trusted

The chronological split removes the worst leakage, but it is still one capture, one lab and one attacker. Three checks show how much of the 0.95 is real.

**Exact copies across the split.** A flood sends the same packet over and over, so a test packet can have an identical copy in the training rows even when it was sent later. For those packets the test measures memory, not detection.

In [ ]:
key = lambda df: pd.util.hash_pandas_object(encode(df[HEADER]).fillna(-999), index=False)
train, test = chrono
train_keys = set(key(train))
has_copy = key(test).isin(train_keys).values

copies = pd.DataFrame({
    "test packets": test.groupby("label_name").size(),
    "% with an exact copy in training": pd.Series(has_copy, index=test.index).groupby(test["label_name"]).mean() * 100,
}).loc[LABELS]
copies.round(1)

Most SYN flood test packets and more than half of the ARP storm test packets already appear, byte for byte in these fields, in the training rows. Here is the same model scored only on test packets **without** an exact copy in training:

In [ ]:
model = gbm().fit(encode(train[HEADER]), train["label"])
pred = model.predict(encode(test[HEADER]))
y = test["label"].values

def scores(mask, name):
    rec = recall_score(y[mask], pred[mask], average=None, labels=range(5), zero_division=0)
    return {"test rows": name, "packets": int(mask.sum()),
            "macro_f1": f1_score(y[mask], pred[mask], average="macro", labels=range(5), zero_division=0),
            **{f"recall_{c}": r for c, r in zip(LABELS[1:], rec[1:])}}

table5 = pd.DataFrame([scores(np.ones(len(y), bool), "all"),
                       scores(~has_copy, "no exact copy in training")]).set_index("test rows")
table5.round(3)

The score drops, and almost all of the drop is ARP storm: the packets the model has not seen before are exactly the ones it struggles with. That is the number to improve, and it is the one a report should show next to the headline score.

**False alarms.** Macro-F1 weighs every class equally, so it hides how often normal traffic is flagged. In production that rate, multiplied by the traffic volume, is the alert load an analyst has to read.

In [ ]:
normal = y == 0
false_alarms = int((pred[normal] != 0).sum())
rate = false_alarms / normal.sum()
print(f"{false_alarms} of {normal.sum():,} normal test packets flagged as an attack ({rate:.2%})")
print(f"at one million normal packets a day: about {rate * 1_000_000:,.0f} false alerts a day")

**Run-to-run noise.** The gap between the random and the chronological split is about one point of macro-F1. Retraining with different seeds shows whether that gap is real or noise:

In [ ]:
rows = []
for seed in range(3):
    for split in SPLITS:
        tr, te = chrono if split == "chronological" else train_test_split(
            full, test_size=0.2, stratify=full["label"], random_state=seed)
        m = HistGradientBoostingClassifier(class_weight="balanced", random_state=seed)
        m.fit(encode(tr[HEADER]), tr["label"])
        rows.append({"split": split, "seed": seed,
                     "macro_f1": f1_score(te["label"], m.predict(encode(te[HEADER])), average="macro")})
noise = pd.DataFrame(rows).groupby("split")["macro_f1"].agg(["mean", "std"])
noise.round(4)

The spread across seeds is far smaller than the gap between the splits, so the gap is real. It is also small, which is the point of section 3: the overall score barely moves, while one class quietly gets worse.

## Summary

In [ ]:
summary = pd.DataFrame([
    {"setting": "always 'Normal' (accuracy)", "score": dummy["accuracy"]},
    {"setting": "always 'Normal' (macro-F1)", "score": dummy["macro_f1"]},
    {"setting": "header + shortcuts, random split", "score": table2.loc[("header + shortcuts", "random"), "macro_f1"]},
    {"setting": "header only, random split", "score": table3.loc["random", "macro_f1"]},
    {"setting": "header only, chronological split", "score": table3.loc["chronological", "macro_f1"]},
    {"setting": "same, test packets without a copy in training", "score": table5.loc["no exact copy in training", "macro_f1"]},
]).set_index("setting")
summary.round(4)

### Checklist for an honest IDS evaluation

1. Report **macro-F1 and per-class recall**, not accuracy.
2. Drop columns that identify the time, the capture position or a specific device, unless the deployment will have them too.
3. Split **by time** (or by capture, network, or day), never by shuffling packets from the same burst.
4. Fit every imputer, scaler and encoder on the **training rows only**, in a pipeline.
5. Check for **exact copies across the split** and report the score without them too.
6. Report the **false-alarm rate on normal traffic**, not only per-class recall.
7. Retrain with a few seeds before calling a difference real.
8. Look at the class that drops: here it is ARP storm, and that is where the next feature or model work should go.
9. Treat every number from a single lab capture as an upper bound until it holds on traffic from a different network.

Questions or a different result on your setup? Leave a comment. The paper behind this dataset is [*Detection of ARP spoofing, SYN and PING flood attacks using machine learning models*](https://infom.fon.bg.ac.rs/index.php/infom/article/view/2678) (Info M, 2024); dataset DOI [10.34740/kaggle/ds/5620663](https://doi.org/10.34740/kaggle/ds/5620663).